Bài tập 1 - Tự viết KNN không dùng thư viện

In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy(dtype=float)
y = df["nhan"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    khoang = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    gan = np.argsort(khoang)[:k]
    so_lop_1 = int(y_hoc[gan].sum())
    return 1 if so_lop_1 > k / 2 else 0

y_pred_tu_viet = [knn_du_doan(X_tr, y_tr, x_moi, 13) for x_moi in X_te]
acc_tu_viet = accuracy_score(y_te, y_pred_tu_viet)

mh_sklearn = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, mh_sklearn.predict(X_te))

print(f"Accuracy tự viết     (k = 13): {acc_tu_viet:.4f}")
print(f"Accuracy scikit-learn (k = 13): {acc_sklearn:.4f}")
print()
print("Giải thích: Hai con số accuracy bằng nhau tuyệt đối (cùng đạt 0.9200) vì hàm knn_du_doan đã thực hiện đúng 3 bước cốt lõi của KNeighborsClassifier: tính khoảng cách Euclid từ mẫu mới tới toàn bộ 225 mẫu của tập học, dùng np.argsort lấy ra 13 mẫu có khoảng cách nhỏ nhất, và đếm số phiếu của lớp 1 (nếu từ 7 phiếu trở lên thì kết luận lớp 1, ngược lại là lớp 0)")

Accuracy tự viết     (k = 13): 0.9200
Accuracy scikit-learn (k = 13): 0.9200

Giải thích: Hai con số accuracy bằng nhau tuyệt đối (cùng đạt 0.9200) vì hàm knn_du_doan đã thực hiện đúng 3 bước cốt lõi của KNeighborsClassifier: tính khoảng cách Euclid từ mẫu mới tới toàn bộ 225 mẫu của tập học, dùng np.argsort lấy ra 13 mẫu có khoảng cách nhỏ nhất, và đếm số phiếu của lớp 1 (nếu từ 7 phiếu trở lên thì kết luận lớp 1, ngược lại là lớp 0)


Bài tập 2 - Khoảng cách Manhattan

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Accuracy trên tập học và tập kiểm tra theo k (Manhattan)")
print("  k   tập học   tập kiểm tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k, metric="manhattan").fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{k:3d}    {a_hoc:.4f}         {a_kt:.4f}")
print()

print("Kiểm định chéo 5 phần trên tập học:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric="manhattan"),
        X_tr, y_tr, cv=5
    ).mean()
k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 27, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  điểm trung bình {diem[k]:.4f}{dau}")
print()

mh = KNeighborsClassifier(n_neighbors=k_tot, metric="manhattan").fit(X_tr, y_tr)
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"k tốt nhất (Manhattan) = {k_tot}")
print(f"Accuracy trên tập kiểm tra (Manhattan) = {acc:.4f}")
print("So với khoảng cách Euclid: k tốt nhất = 13, Accuracy = 0.9200")
print()
print("Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan làm k tốt nhất đổi từ 13 sang 27 (cũng đạt điểm kiểm định chéo 0.8667), nhưng accuracy trên tập kiểm tra chỉ giảm nhẹ từ 0.9200 xuống 0.9067 (chênh lệch 1 mẫu), cho thấy kết quả không khác đi nhiều trên bộ dữ liệu 2 chiều này.")

Accuracy trên tập học và tập kiểm tra theo k (Manhattan)
  k   tập học   tập kiểm tra
  1    1.0000         0.7867
  5    0.8756         0.8933
 13    0.8667         0.9067
 31    0.8711         0.9067
101    0.8311         0.8400

Kiểm định chéo 5 phần trên tập học:
  k =  1  điểm trung bình 0.7822
  k =  5  điểm trung bình 0.8444
  k =  9  điểm trung bình 0.8222
  k = 13  điểm trung bình 0.8489
  k = 17  điểm trung bình 0.8622
  k = 25  điểm trung bình 0.8578
  k = 27  điểm trung bình 0.8667 <- tot nhat
  k = 51  điểm trung bình 0.8489

k tốt nhất (Manhattan) = 27
Accuracy trên tập kiểm tra (Manhattan) = 0.9067
So với khoảng cách Euclid: k tốt nhất = 13, Accuracy = 0.9200

Nhận xét: Đổi cách đo khoảng cách từ Euclid sang Manhattan làm k tốt nhất đổi từ 13 sang 27 (cũng đạt điểm kiểm định chéo 0.8667), nhưng accuracy trên tập kiểm tra chỉ giảm nhẹ từ 0.9200 xuống 0.9067 (chênh lệch 1 mẫu), cho thấy kết quả không khác đi nhiều trên bộ dữ liệu 2 chiều này.


Bài tập 3 - Đổi đơn vị đo

In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
df["chi_tieu_trieu"] = df["chi_tieu"] / 1000.0

X_moi = df[["gio_on", "diem_giua_ky", "chi_tieu_trieu"]]
y = df["qua_mon"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_moi, y, test_size=0.25, random_state=17, stratify=y
)

mh_khong_chuan = KNeighborsClassifier(n_neighbors=7).fit(X_tr, y_tr)
acc_khong_chuan = accuracy_score(y_te, mh_khong_chuan.predict(X_te))

mh_co_chuan = make_pipeline(StandardScaler(), KNeighborsClassifier(7)).fit(X_tr, y_tr)
acc_co_chuan = accuracy_score(y_te, mh_co_chuan.predict(X_te))

print(f"Không chuẩn hóa (chỉ tiêu tính bằng nghìn đồng - Mục 3): 0.5333")
print(f"Không chuẩn hóa (chỉ tiêu tính bằng triệu đồng)        : {acc_khong_chuan:.4f}")
print(f"Có chuẩn hóa    (chỉ tiêu tính bằng triệu đồng)        : {acc_co_chuan:.4f}")
print()
print("Giải thích:")
print("- Khi đổi đơn vị từ nghìn đồng sang triệu đồng (chia 1000), accuracy khi KHÔNG chuẩn hóa tăng vọt từ 0.5333 lên 0.8667.")
print("- Nguyên nhân: Ở đơn vị nghìn đồng, độ lệch chuẩn của chi_tieu là 1692, lớn gấp hàng trăm lần gio_on (7.82) và diem_giua_ky (2.09), nên cột chi_tieu nuốt trọn phép tính khoảng cách Euclid. Khi chia cho 1000 sang triệu đồng, độ lệch chuẩn của chi_tieu giảm xuống còn 1.69, nhỏ hơn cả gio_on và diem_giua_ky, nhờ vậy hai cột học tập lấy lại được quyền quyết định khoảng cách.")
print("- Nếu CÓ chuẩn hóa (StandardScaler), việc đổi đơn vị KHÔNG làm kết quả thay đổi (vẫn giữ nguyên 0.8667), vì phép chuẩn hóa z = (x - tb) / dl triệt tiêu hoàn toàn hệ số nhân 1/1000 ở cả tử số và mẫu số.")

Không chuẩn hóa (chỉ tiêu tính bằng nghìn đồng - Mục 3): 0.5333
Không chuẩn hóa (chỉ tiêu tính bằng triệu đồng)        : 0.8667
Có chuẩn hóa    (chỉ tiêu tính bằng triệu đồng)        : 0.8667

Giải thích:
- Khi đổi đơn vị từ nghìn đồng sang triệu đồng (chia 1000), accuracy khi KHÔNG chuẩn hóa tăng vọt từ 0.5333 lên 0.8667.
- Nguyên nhân: Ở đơn vị nghìn đồng, độ lệch chuẩn của chi_tieu là 1692, lớn gấp hàng trăm lần gio_on (7.82) và diem_giua_ky (2.09), nên cột chi_tieu nuốt trọn phép tính khoảng cách Euclid. Khi chia cho 1000 sang triệu đồng, độ lệch chuẩn của chi_tieu giảm xuống còn 1.69, nhỏ hơn cả gio_on và diem_giua_ky, nhờ vậy hai cột học tập lấy lại được quyền quyết định khoảng cách.
- Nếu CÓ chuẩn hóa (StandardScaler), việc đổi đơn vị KHÔNG làm kết quả thay đổi (vẫn giữ nguyên 0.8667), vì phép chuẩn hóa z = (x - tb) / dl triệt tiêu hoàn toàn hệ số nhân 1/1000 ở cả tử số và mẫu số.


Bài tập 4 - Láng giềng có trọng số

In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("So sánh KNN không trọng số (uniform) và có trọng số (distance) trên tập kiểm tra:")
print("  k   uniform (Mục 4)   distance")
for k in [1, 13, 101]:
    mh_deu = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    mh_kc = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    acc_deu = accuracy_score(y_te, mh_deu.predict(X_te))
    acc_kc = accuracy_score(y_te, mh_kc.predict(X_te))
    print(f"{k:3d}            {acc_deu:.4f}     {acc_kc:.4f}")

print()
print("Giải thích: Với k = 101, trọng số theo khoảng cách GIÚP mô hình bớt cẩu thả rõ rệt (accuracy trên tập kiểm tra tăng từ 0.8533 lên 0.9067). Vì khi dùng weights='distance', mỗi láng giềng được nhân trọng số bằng nghịch đảo khoảng cách (1/d); dù ta lấy tới 101 láng giềng, nhưng các điểm ở xa có khoảng cách d lớn nên phiếu bầu rất nhẹ, trong khi những láng giềng nằm sát mẫu mới có phiếu bầu rất nặng và giữ quyền quyết định, giúp biên quyết định vẫn bám theo hình dáng cong của dữ liệu.")

So sánh KNN không trọng số (uniform) và có trọng số (distance) trên tập kiểm tra:
  k   uniform (Mục 4)   distance
  1            0.8000     0.8000
 13            0.9200     0.9333
101            0.8533     0.9067

Giải thích: Với k = 101, trọng số theo khoảng cách GIÚP mô hình bớt cẩu thả rõ rệt (accuracy trên tập kiểm tra tăng từ 0.8533 lên 0.9067). Vì khi dùng weights='distance', mỗi láng giềng được nhân trọng số bằng nghịch đảo khoảng cách (1/d); dù ta lấy tới 101 láng giềng, nhưng các điểm ở xa có khoảng cách d lớn nên phiếu bầu rất nhẹ, trong khi những láng giềng nằm sát mẫu mới có phiếu bầu rất nặng và giữ quyền quyết định, giúp biên quyết định vẫn bám theo hình dáng cong của dữ liệu.


Bài tập 5 - Chọn $k$ cho bài toán hồi quy

In [13]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print("Kiểm định chéo 5 phần trên tập học (MAE):")
diem_mae = {}
for k in range(1, 21):
    diem_am = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error"
    )
    diem_mae[k] = -diem_am.mean()
    print(f"  k = {k:2d}  MAE trung bình = {diem_mae[k]:.4f}")

k_tot = min(diem_mae, key=diem_mae.get)
print()
print(f"k tốt nhất = {k_tot} (MAE kiểm định chéo = {diem_mae[k_tot]:.4f})")

knn_tot = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

mae_knn = mean_absolute_error(y_te, knn_tot.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f"MAE trên tập kiểm tra của KNN (k = {k_tot}): {mae_knn:.4f} tỷ đồng")
print(f"MAE trên tập kiểm tra của đường thẳng  : {mae_duong:.4f} tỷ đồng")
print()
print("Nhận xét: Với k tốt nhất = 2, MAE trên tập kiểm tra của KNN đạt 0.4867 tỷ đồng, vẫn KHÔNG vượt được mô hình đường thẳng (MAE = 0.2578 tỷ đồng). Lý do là quan hệ giữa diện tích và giá nhà trong tập dữ liệu này thực sự tuân theo quy luật tuyến tính, nên mô hình hồi quy tuyến tính nắm bắt quy luật toàn cục chính xác hơn việc lấy trung bình cục bộ của KNN.")

Kiểm định chéo 5 phần trên tập học (MAE):
  k =  1  MAE trung bình = 0.4164
  k =  2  MAE trung bình = 0.3498
  k =  3  MAE trung bình = 0.3714
  k =  4  MAE trung bình = 0.3618
  k =  5  MAE trung bình = 0.3816
  k =  6  MAE trung bình = 0.4002
  k =  7  MAE trung bình = 0.3990
  k =  8  MAE trung bình = 0.4098
  k =  9  MAE trung bình = 0.4171
  k = 10  MAE trung bình = 0.4365
  k = 11  MAE trung bình = 0.4500
  k = 12  MAE trung bình = 0.4695
  k = 13  MAE trung bình = 0.4992
  k = 14  MAE trung bình = 0.5153
  k = 15  MAE trung bình = 0.5290
  k = 16  MAE trung bình = 0.5595
  k = 17  MAE trung bình = 0.5748
  k = 18  MAE trung bình = 0.5891
  k = 19  MAE trung bình = 0.6130
  k = 20  MAE trung bình = 0.6254

k tốt nhất = 2 (MAE kiểm định chéo = 0.3498)
MAE trên tập kiểm tra của KNN (k = 2): 0.4867 tỷ đồng
MAE trên tập kiểm tra của đường thẳng  : 0.2578 tỷ đồng

Nhận xét: Với k tốt nhất = 2, MAE trên tập kiểm tra của KNN đạt 0.4867 tỷ đồng, vẫn KHÔNG vượt được mô hình đường thẳng (

Bài tập 6 - Đo thời gian dự đoán

In [14]:
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))

print(f"{'Số mẫu học':>12s} {'Thời gian fit (s)':>20s} {'Thời gian predict (s)':>24s}")
for n in [1000, 10000, 100000]:
    X_hoc = rng.random((n, 2))
    y_hoc = rng.integers(0, 2, size=n)
    
    mh = KNeighborsClassifier(n_neighbors=5, algorithm="brute")
    
    t0 = time.time()
    mh.fit(X_hoc, y_hoc)
    t_fit = time.time() - t0
    
    t0 = time.time()
    mh.predict(X_moi)
    t_pred = time.time() - t0
    
    print(f"{n:12d} {t_fit:20.6f} {t_pred:24.6f}")

print()
print("Giải thích: KNN được gọi là mô hình học lười (lazy learning) vì ở bước huấn luyện (.fit), mô hình gần như không tính toán gì mà chỉ cất dữ liệu vào bộ nhớ nên thời gian fit rất nhỏ. Ngược lại, mọi khối lượng tính toán nặng nề đều dồn sang lúc dự đoán (.predict), khi mô hình phải tính khoảng cách từ 1000 mẫu mới tới toàn bộ các mẫu trong tập học; do đó khi kích thước tập học tăng từ 1000 lên 100000 mẫu, thời gian predict tăng lên rõ rệt.")

  Số mẫu học    Thời gian fit (s)    Thời gian predict (s)
        1000             0.000749                 0.002447
       10000             0.000452                 0.004857
      100000             0.001997                 0.028413

Giải thích: KNN được gọi là mô hình học lười (lazy learning) vì ở bước huấn luyện (.fit), mô hình gần như không tính toán gì mà chỉ cất dữ liệu vào bộ nhớ nên thời gian fit rất nhỏ. Ngược lại, mọi khối lượng tính toán nặng nề đều dồn sang lúc dự đoán (.predict), khi mô hình phải tính khoảng cách từ 1000 mẫu mới tới toàn bộ các mẫu trong tập học; do đó khi kích thước tập học tăng từ 1000 lên 100000 mẫu, thời gian predict tăng lên rõ rệt.
